In [1]:
import nibabel as nib
import sys
sys.path.append('../')
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # this MUST come before any tf call.

from brain_age.config import Config
from brain_age.data import dataset_manager
from pathlib import Path
from brain_age.data.augmentation import get_augmenter
from brain_age.python_utils import load_args_config, findGPUtoUse, log_every_packaged_loaded, prepare_yaml_for_saving
import tensorflow as tf
from loguru import logger

/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (2.2.0) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [25]:


config = {'data': {
    'Path_in_csv': '/brain_age/data/analyses/',
    'Filename_csv': 'LDM_10k_ss_preproc_full_ba.csv',#'LDM_100k_ss_mars.csv',
    'Filename_weights_csv': 'sample_weights.csv',
    'predicted_variable': 'PatientAge',
    'pad_vols': True,
    'orient_vols': 'LIA',
    'Inh_vol_path': '/brain_age/data/analyses/inhomogeneity_volume/inhomogeneity.npy',
    'Path_in_data': '/brain_age/data/'
}, 'network': {
    'num_classes': 1,
    'shape': 256,
    'use_se': False,
    'se_ratio': 1.0,
    'activation': 'elu',
    'bn': 'BN', 
    'conv_block': 'BottleNeck',
    'dropout_rate': 0.001,
    'final_stage': 'avgPool',
    'num_conv_layers': 2,
    'num_initial_filter': 32, 
    'identity_layers': False, 
    'n_identity_layers': 2,
}, 'experiment': { 
    'training_path': '/tmp/',
    'num_gpus': 4,
}, 'training': {
    'sample_weights': 'default',
    'hemisphere': 'full',
    'batch_size': 8,
    'loss_weights_age': 0.25,
    'lr': 0.001,
    'loss_weights': True,
    'lr_decay': -0.01,
    'optimizer': 'Adam',
    'profile': False,
    'epochs': 20
}, 'augment': {
    'augmentation': True,
    'prob_overall': 0.9,
    'prob_geom': 0.5,
    'prob_colo': 0.5,
    'prob_tran': 1,
    'prob_rota': 2,
    'prob_inho': 0.7,
    'prob_blur': 0.5,
    'prob_salt': 0.5,
    'prob_gaus': 0.3,
    'prob_down': 0.3,
    'prob_gamm': 0.3,
    'prob_cont': 1,
    'prob_slic': 0.3,
    'prob_bias': 2,
    'prob_moti': 2,
    'prob_ghost': 1
}, 'testing': {
    'size': 1
}}


In [26]:

config = Config(**config)
config.experiment.training_path = Path(config.experiment.training_path)
# config.network.shape = [config.network.shape] * 3
# config.network.num_classes = [1]
dataset = dataset_manager.prepareDataset(config, save_csv_copy=False)
    
# augmenter = get_augmenter(dataset['inhomogeneity_volume'], config)


2024-03-08 21:03:40.924 | DEBUG    | brain_age.data.dataset_manager:_load_csv:68 - data.predicted_variable set to ['PatientAge']
/brain_age/src/data_analysis/../brain_age/data/dataset_manager.py:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_set['mask'] = df_set['mask'].fillna('')
/brain_age/src/data_analysis/../brain_age/data/dataset_manager.py:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_set['mask'] = df_set['mask'].fillna('')
/brain_age/src/data_analysis/../brain_age/data/dataset_manager.py

In [7]:
gpus = tf.config.experimental.list_physical_devices('GPU')
print(gpus)
gpus_used = findGPUtoUse(config.experiment.num_gpus)

2024-03-08 17:57:42.821 | DEBUG    | brain_age.python_utils:selectGPUsAvailability:85 - GPU:3 memory usage: 0.0GB
2024-03-08 17:57:42.822 | DEBUG    | brain_age.python_utils:selectGPUsAvailability:85 - GPU:2 memory usage: 0.0GB
2024-03-08 17:57:42.823 | DEBUG    | brain_age.python_utils:selectGPUsAvailability:85 - GPU:1 memory usage: 0.0GB
2024-03-08 17:57:42.824 | DEBUG    | brain_age.python_utils:selectGPUsAvailability:85 - GPU:0 memory usage: 0.0GB
2024-03-08 17:57:42.825 | INFO     | brain_age.python_utils:findGPUtoUse:107 - Setting visible GPUS="[(PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU'), 'GPU:3'), (PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), 'GPU:2'), (PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), 'GPU:1'), (PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), 'GPU:0')]"
2024-03-08 17:57:42.826 | INFO     | brain_age.python_utils:findGPUtoUse:112 - GPU used: [(PhysicalDevice(name='/physical_device:GPU:

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]


In [ ]:

# if config.experiment.num_gpus > 1:
# strategy = tf.distribute.MirroredStrategy(devices=[device for _, device in gpus_used])
# global_batch_size = config.training.batch_size
# batch_size_per_replica = global_batch_size // strategy.num_replicas_in_sync
# logger.info(f'Using {config.experiment.num_gpus} GPUs to train, with a global batch size of {global_batch_size}. Each replica will have a batch size of {batch_size_per_replica}.')
# strategy.experimental_distribute_dataset(ds_train)
# strategy.experimental_distribute_dataset(ds_valid)
# strategy.experimental_distribute_dataset(ds_test)

# Temporarily set batch_size to only per replica size
# config.training.batch_size = batch_size_per_replica

# if config.experiment.num_gpus > 1:
#     strategy = tf.distribute.MirroredStrategy(devices=[device for _, device in gpus_used])
#     global_batch_size = config.training.batch_size
#     batch_size_per_replica = global_batch_size // strategy.num_replicas_in_sync
#     logger.info(f'Using {config.experiment.num_gpus} GPUs to train, with a global batch size of {global_batch_size}. Each replica will have a batch size of {batch_size_per_replica}.')
#     # strategy.experimental_distribute_dataset(ds_train)
#     # strategy.experimental_distribute_dataset(ds_valid)
#     # strategy.experimental_distribute_dataset(ds_test)

#     ds_train, ds_valid, ds_test = dataset_manager.TFDatasetGeneratorDistributed(config, strategy, dataset, augmenter)
#     ds_data = {'train': ds_train, 'valid': ds_valid, 'test': ds_test}
# else:
#     # This will default to a single GPU/CPU strategy
#     strategy = tf.distribute.get_strategy()

#     ds_train, ds_valid, ds_test = dataset_manager.TFDatasetGenerator(config, dataset, augmenter)
#     ds_data = {'train': ds_train, 'valid': ds_valid, 'test': ds_test}


In [ ]:
# distributed_dataset_iterator = iter(ds_valid)
# for _ in range(1):
#     x, y, w = next(distributed_dataset_iterator)
#     print(x)
#     print(y)
#     print(w)

In [ ]:
# # Assuming 'strategy' is your tf.distribute.Strategy instance and 'dist_dataset' is your distributed dataset
# # def print_shapes(replica_inputs):
# #     x, y, w = replica_inputs
# #     # Use strategy.experimental_local_results to convert PerReplica objects to Tensors
# #     x_tensors = tf.nest.map_structure(strategy.experimental_local_results, x)
# #     y_tensors = tf.nest.map_structure(strategy.experimental_local_results, y)
# #     w_tensors = tf.nest.map_structure(strategy.experimental_local_results, w)
    
# #     for i, tensor in enumerate(x_tensors[0]):
# #         print(f"Replica {i}, X shape: {tensor.shape}")
# #     for i, tensor in enumerate(y_tensors[0]):
# #         print(f"Replica {i}, Y shape: {tensor.shape}")
# #     for i, tensor in enumerate(w_tensors[0]):
# #         print(f"Replica {i}, W shape: {tensor.shape}")

# # distributed_dataset_iterator = iter(ds_train)
# # for _ in range(1):
# #     strategy.run(print_shapes, args=(next(distributed_dataset_iterator),))

# @tf.function
# def train_step(inputs):
#   features, labels, weights = inputs
#   return labels - 0.3 * features

# for _ in range(1):
#   # train_step trains the model using the dataset elements
#   loss = strategy.run(train_step, args=(next(distributed_dataset_iterator),))
#   print("Loss is ", loss)


In [ ]:
# from brain_age.model.training import train

# with strategy.scope():
#     logger.info("\n\n\n******** Training ********")
#     model = train(config, ds_valid, ds_valid)

## Simple example using toy data and toy model

In [ ]:
# One actual, working example :')

strategy = tf.distribute.MirroredStrategy(devices=[device for _, device in gpus_used],
                                         cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())

global_batch_size=16
dataset = tf.data.Dataset.from_tensors(([1.], [1.])).repeat(64).shuffle(2).batch(global_batch_size).prefetch(2)
train_dataset=dataset
eval_dataset = tf.data.Dataset.from_tensors(([0.], [0.])).repeat(64).batch(global_batch_size).prefetch(2)

class CustomTrainingLogger(tf.keras.callbacks.Callback):
    
    def on_epoch_begin(self, epoch, logs=None):
        print(f'Starting epoch {epoch}')
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss')
        val_loss = logs.get('val_loss')
        print(f"End of epoch {epoch + 1}: loss = {loss}, val_loss = {val_loss}")
        # Add more metrics if needed, e.g., accuracy:
        accuracy = logs.get('accuracy')
        val_accuracy = logs.get('val_accuracy')
        if accuracy is not None:
            print(f"accuracy = {accuracy}, val_accuracy = {val_accuracy}")

            
with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(1,)),  # Assuming input features of size 10
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')  # Assuming binary classification
    ])

    # Compile the model (necessary for training, but not for a simple forward pass)
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
model.fit(dataset, validation_data = eval_dataset,
              validation_steps = 4, epochs=1, steps_per_epoch=4, callbacks=[CustomTrainingLogger()])
    
# def dataset_fn(input_context):
#     batch_size = input_context.get_per_replica_batch_size(global_batch_size)
#     dataset = tf.data.Dataset.from_tensors(([1.], [1.])).repeat(64)
#     dataset = dataset.shard(
#       input_context.num_input_pipelines, input_context.input_pipeline_id)
#     dataset = dataset.batch(batch_size)
#     dataset = dataset.prefetch(2)  # This prefetches 2 batches per device.
#     return dataset

# dist_dataset = strategy.distribute_datasets_from_function(dataset_fn)

# @tf.function
# def train_step(inputs):
#     features, labels = inputs
#     return labels - 0.3 * features

# for x in dist_dataset:
#     # train_step trains the model using the dataset elements
#     loss = strategy.run(train_step, args=(x,))
#     print("Loss is ", loss)


## Simple example using a small model with 256^3 toy tensors

In [ ]:
# One actual, working example :')

from brain_age.model.network import get_model

# Due to a bug https://github.com/tensorflow/tensorflow/issues/54325 we need to use something other than NCCL reduce
strategy = tf.distribute.MirroredStrategy(devices=[device for _, device in gpus_used],
                                         cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())

def generate_zeros(*args):
    return tf.zeros((256, 256, 256, 1)), tf.zeros((1,))  # Assuming you want labels to be a tensor of shape (1,)

global_batch_size=2
dataset = tf.data.Dataset.from_tensors(([0.], [0.])).repeat(64).shuffle(2).map(lambda x, y: generate_zeros(x, y)).batch(global_batch_size).prefetch(2)
train_dataset=dataset

eval_dataset = tf.data.Dataset.from_tensors(([0.], [0.])).repeat(64).map(lambda x, y: generate_zeros(x, y)).batch(global_batch_size).prefetch(2)

with strategy.scope():
    model = get_model(config)
    
    opt = tf.keras.optimizers.get(config.training.optimizer)
    opt.lr.assign(config.training.lr)
    
    model.compile(optimizer = opt,
                loss = tf.keras.losses.mean_squared_error,#losses.losses_dict[loss],
#                 metrics = metrics, # jit_compile=False,
#                 loss_weights = loss_weights_dict
                )

class CustomTrainingLogger(tf.keras.callbacks.Callback):
    
    def on_epoch_begin(self, epoch, logs=None):
        print(f'Starting epoch {epoch}')
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss')
        val_loss = logs.get('val_loss')
        print(f"End of epoch {epoch + 1}: loss = {loss}, val_loss = {val_loss}")
        # Add more metrics if needed, e.g., accuracy:
        accuracy = logs.get('accuracy')
        val_accuracy = logs.get('val_accuracy')
        if accuracy is not None:
            print(f"accuracy = {accuracy}, val_accuracy = {val_accuracy}")
            
    
model.fit(dataset, validation_data = eval_dataset,
              validation_steps = 2, epochs=1, steps_per_epoch=2, callbacks=[CustomTrainingLogger()])


## Example using fixed inputs and 50% 1 and 2 labels to predict mean value

In [11]:
import numpy as np

from brain_age.model.network import get_model

# Due to a bug https://github.com/tensorflow/tensorflow/issues/54325 we need to use something other than NCCL reduce
strategy = tf.distribute.MirroredStrategy(devices=[device for _, device in gpus_used],
                                         cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())


options = tf.data.Options()
options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA

def generate_alternating(*args):
    def generator():
        while True:
            yield tf.ones((256, 256, 256, 1)), tf.zeros((1,))
            yield tf.ones((256, 256, 256, 1)), tf.ones((1,))
    return tf.data.Dataset.from_generator(generator, output_types=(tf.float32, tf.float32), output_shapes=((256, 256, 256, 1), (1,)))

global_batch_size=8
dataset = generate_alternating().batch(global_batch_size).prefetch(2).with_options(options)
train_dataset = dataset
eval_dataset = generate_alternating().batch(global_batch_size).prefetch(2).with_options(options)

config.training.lr=0.001

# print(next(iter(train_dataset))[1])

with strategy.scope():
    model = get_model(config)
    
    opt = tf.keras.optimizers.get('SGD')
    opt.lr.assign(config.training.lr)
    
    model.compile(optimizer = opt,
                loss = tf.keras.losses.mean_absolute_error,#losses.losses_dict[loss],
#                 metrics = metrics, # jit_compile=False,
#                 loss_weights = loss_weights_dict
                )

class CustomTrainingLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss')
        val_loss = logs.get('val_loss')
        print(f"End of epoch {epoch + 1}: loss = {loss}, val_loss = {val_loss}")

        # Calculate and log the mean model output value over the validation dataset
        mean_pred = self.calculate_mean_prediction(self.model, train_dataset, steps=5)
        print(f"Mean model output over training dataset at epoch {epoch + 1}: {mean_pred}")
        mean_pred = self.calculate_mean_prediction(self.model, eval_dataset, steps=5)
        print(f"Mean model output over validation dataset at epoch {epoch + 1}: {mean_pred}")

    def calculate_mean_prediction(self, model, dataset, steps):
        # Collect predictions using model.predict, which runs in inference mode
        predictions = model.predict(dataset, steps=steps)
        mean_pred = np.mean(predictions)
        return mean_pred
    
model.fit(train_dataset, validation_data=train_dataset,
          epochs=50, steps_per_epoch=5, validation_steps=5,
          callbacks=[CustomTrainingLogger()])

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:3', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:0')
Epoch 1/50
INFO:tensorflow:batch_all_reduce: 18 all-reduces with algorithm = hierarchical_copy, num_packs = 1
INFO:tensorflow:batch_all_reduce: 18 all-reduces with algorithm = hierarchical_copy, num_packs = 1
5/5 [==============================] - 4s 509ms/step
Mean model output over training dataset at epoch 1: -1.0849909782409668
5/5 [==============================] - 3s 502ms/step
Mean model output over validation dataset at epoch 1: -1.0849909782409668
5/5 [==============================] - 22s 4s/step - loss: 1.6892 - val_loss: 1.6077
Epoch 2/50
5/5 [==============================] - 3s 502ms/step
Mean model output over training dataset at epoch 2: -0.9489213824272156
5/5 [==============================] - 3s 501ms/step
Mean model output ov

Epoch 17/50
5/5 [==============================] - 3s 517ms/step
Mean model output over training dataset at epoch 17: 0.017117660492658615
5/5 [==============================] - 3s 500ms/step
Mean model output over validation dataset at epoch 17: 0.017117660492658615
5/5 [==============================] - 13s 3s/step - loss: 0.5227 - val_loss: 0.5227
Epoch 18/50
5/5 [==============================] - 3s 503ms/step
Mean model output over training dataset at epoch 18: 0.017140131443738937
5/5 [==============================] - 3s 509ms/step
Mean model output over validation dataset at epoch 18: 0.017140131443738937
5/5 [==============================] - 13s 3s/step - loss: 0.5227 - val_loss: 0.5227
Epoch 19/50
5/5 [==============================] - 3s 513ms/step
Mean model output over training dataset at epoch 19: 0.017134379595518112
5/5 [==============================] - 3s 508ms/step
Mean model output over validation dataset at epoch 19: 0.017134379595518112
5/5 [=====================

KeyboardInterrupt: 

In [12]:
def calculate_mean_prediction(model, dataset, steps):
    # Collect predictions using model.predict, which runs in inference mode
    predictions = model.predict(dataset, steps=steps)
    mean_pred = np.mean(predictions)
    return mean_pred
print(calculate_mean_prediction(model, eval_dataset, steps=5))

5/5 [==============================] - 3s 506ms/step
0.017132293


## Basic example using mostly our usual code

In [27]:
# One actual, working example :')

from brain_age.model.network import get_model
from brain_age.model import losses

config.network.shape = 256

# Due to a bug https://github.com/tensorflow/tensorflow/issues/54325 we need to use something other than NCCL reduce
strategy = tf.distribute.MirroredStrategy(devices=[device for _, device in gpus_used],
                                         cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())

ds_train, ds_valid, ds_test = dataset_manager.TFDatasetGenerator(config, dataset, None)
ds_data = {'train': ds_train, 'valid': ds_valid, 'test': ds_test}

# Deal with losses and metrics
loss = {y:(losses.losses_dict[config.training.loss] if x == 1 else 'categorical_crossentropy') for x,y in zip(config.network.num_classes,config.data.predicted_variable)}
metrics = {y:(list(losses.metrics_dict.values()) if x == 1 else [losses.metrics_dict['accuracy']]) for x,y in zip(config.network.num_classes,config.data.predicted_variable)}

with strategy.scope():
    model = get_model(config)
    
    opt = tf.keras.optimizers.get(config.training.optimizer)
    opt.lr.assign(config.training.lr)
    
    model.compile(optimizer = opt,
                    loss = loss, #losses.losses_dict[loss],
                    metrics = metrics, # jit_compile=False,
#                     loss_weights = loss_weights_dict
                    )

class CustomTrainingLogger(tf.keras.callbacks.Callback):
    
    def on_epoch_begin(self, epoch, logs=None):
        print(f'Starting epoch {epoch}')
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss')
        val_loss = logs.get('val_loss')
        print(f"End of epoch {epoch + 1}: loss = {loss}, val_loss = {val_loss}")
        # Add more metrics if needed, e.g., accuracy:
        accuracy = logs.get('accuracy')
        val_accuracy = logs.get('val_accuracy')
        if accuracy is not None:
            print(f"accuracy = {accuracy}, val_accuracy = {val_accuracy}")
    

def train_model(config: Config, model, ds_train, ds_valid, wand=True):
    """
    Train a single pre-instantiated model
    :param config: config object
    :param model: tf model
    :param ds_train: tf.data
    :param ds_valid: tf.data
    """

    model.summary(print_fn=lambda x: print(x), line_length=80)           # log the summary of the model
    STEPS_PER_EPOCH = config.training.train_size // config.training.batch_size
    
    # Learning rate scheduler and logger   
    def lr_scheduler(epoch, lr):
        return max(lr * tf.math.exp(config.training.lr_decay * epoch), 1e-5)
    reduce_lr = tf.keras.callbacks.LearningRateScheduler(lr_scheduler)
    # reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=3, min_lr=1e-7, verbose=1)
    
    callbacks = [reduce_lr, CustomTrainingLogger()]

    # Training loop
    model.fit(ds_train,
              steps_per_epoch = config.training.train_size // config.training.batch_size,
              epochs = config.training.epochs,
              validation_data = ds_valid,
              validation_steps = config.training.validation_steps // config.training.batch_size, # "validation steps" is the number of batches. We want our "validation_steps" param to be the number of volumes. It's easier to track.
              validation_freq = config.training.validation_freq,
              callbacks = callbacks,    
              use_multiprocessing = True,
              workers = 8,
              verbose = 2,
              )    

# model.fit(dataset, validation_data = eval_dataset,
#               validation_steps = 2, epochs=1, steps_per_epoch=2, callbacks=[CustomTrainingLogger()])


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:3', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:0')


2024-03-08 21:03:54.821 | INFO     | brain_age.data.dataset_manager:TFDatasetGenerator:304 - config.network.shape is set to [256, 256, 256]
2024-03-08 21:03:54.822 | WARNING  | brain_age.data.dataset_manager:TFDatasetGenerator:307 - --augment.augmentation is True, but passed Augmenter is None. Is that right?


In [28]:
train_model(config, model, ds_train, ds_valid, wand=False)

Model: "model_3"
________________________________________________________________________________
 Layer (type)                       Output Shape                    Param #     
 input_4 (InputLayer)               [(None, 256, 256, 256, 1)]      0           
                                                                                
 enc_conv_1_1 (BottleNeck)          (None, 128, 128, 128, 32)       7752        
                                                                                
 enc_conv_2_1 (BottleNeck)          (None, 64, 64, 64, 64)          33808       
                                                                                
 GlobalAveragePooling_1 (GlobalAve  (None, 64)                      0           
 ragePooling3D)                                                                 
                                                                                
 multihead_output (Layer)           (None, 64)                      0           
           

End of epoch 9: loss = 0.9646872282028198, val_loss = 1.0112720727920532
1149/1149 - 1221s - loss: 0.9647 - categorical_crossentropy: -1.6091e-10 - mean_squared_error: 0.9484 - mean_absolute_error: 0.8381 - log_cosh: 0.3816 - categorical_accuracy: 1.0000 - val_loss: 1.0113 - val_categorical_crossentropy: 7.9088e-10 - val_mean_squared_error: 0.9954 - val_mean_absolute_error: 0.8523 - val_log_cosh: 0.3913 - val_categorical_accuracy: 1.0000 - lr: 6.9768e-04 - 1221s/epoch - 1s/step
Starting epoch 9
Epoch 10/20
End of epoch 10: loss = 0.9551169276237488, val_loss = 3.349306344985962
1149/1149 - 1147s - loss: 0.9551 - categorical_crossentropy: 2.9922e-11 - mean_squared_error: 0.9394 - mean_absolute_error: 0.8326 - log_cosh: 0.3783 - categorical_accuracy: 1.0000 - val_loss: 3.3493 - val_categorical_crossentropy: 7.9088e-10 - val_mean_squared_error: 3.3338 - val_mean_absolute_error: 1.5884 - val_log_cosh: 1.0193 - val_categorical_accuracy: 1.0000 - lr: 6.3763e-04 - 1147s/epoch - 999ms/step
Sta

Starting epoch 19
Epoch 20/20
End of epoch 20: loss = 0.8071809411048889, val_loss = 4.61359167098999
1149/1149 - 1133s - loss: 0.8072 - categorical_crossentropy: 3.8526e-11 - mean_squared_error: 0.7915 - mean_absolute_error: 0.7550 - log_cosh: 0.3240 - categorical_accuracy: 1.0000 - val_loss: 4.6136 - val_categorical_crossentropy: 7.9088e-10 - val_mean_squared_error: 4.5979 - val_mean_absolute_error: 1.9973 - val_log_cosh: 1.3501 - val_categorical_accuracy: 1.0000 - lr: 1.4957e-04 - 1133s/epoch - 986ms/step
